In [1]:
import asyncio
import inspect
import json
import os
import re
import threading
import uuid
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints
from zoneinfo import ZoneInfo

import anthropic
import numpy as np
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from mcp import Client
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [6]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

In [7]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls]}

In [8]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

In [9]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])])
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [10]:
system_message = SystemMessage("You're an AI assistant.")

In [11]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)

In [12]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )

In [13]:
@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


print(json.dumps(add_days.parameters, indent=2))
print(add_days("2026-11-10", 100))

{
  "type": "object",
  "properties": {
    "date": {
      "type": "string",
      "description": "Start date in YYYY-MM-DD format"
    },
    "days": {
      "type": "integer",
      "description": "Number of days to add, negative to subtract"
    }
  },
  "required": [
    "date"
  ]
}
2027-02-18 (Thursday)


In [14]:
class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it

In [15]:
class MCPConnection:
    """A synchronous wrapper around the async mcp.Client (Part 6 will make the whole Agent async).

    The connection lives in one long-running coroutine on a background event loop;
    every call is submitted to that loop and we wait for its result.
    """

    def __init__(self, server):  # StdioServerParameters, or the URL of a remote server
        self.loop = asyncio.new_event_loop()
        threading.Thread(target=self.loop.run_forever, daemon=True).start()
        ready = threading.Event()
        self.session = asyncio.run_coroutine_threadsafe(self._session(server, ready), self.loop)
        ready.wait()
        if self.session.done():  # connecting failed: raise the error here
            self.session.result()

    async def _session(self, server, ready: threading.Event):
        self.closed = asyncio.Event()
        try:
            async with Client(server) as client:
                self.client = client
                ready.set()
                await self.closed.wait()  # keep the connection open until close()
        finally:
            ready.set()

    def _run(self, coroutine):
        return asyncio.run_coroutine_threadsafe(coroutine, self.loop).result()

    def list_tools(self):
        return self._run(self.client.list_tools()).tools

    def call_tool(self, name: str, arguments: dict):
        return self._run(self.client.call_tool(name, arguments))

    def close(self):
        self.loop.call_soon_threadsafe(self.closed.set)
        self.session.result()
        self.loop.call_soon_threadsafe(self.loop.stop)

In [16]:
class MCPTool(Tool):
    """A tool that lives on an MCP server: same interface as Tool, but run() goes over MCP."""

    def __init__(self, connection: MCPConnection, mcp_tool):
        super().__init__(mcp_tool.name, mcp_tool.description or "", mcp_tool.input_schema, function=None)
        self.connection = connection

    def run(self, arguments: dict) -> str:
        result = self.connection.call_tool(self.name, arguments)
        text = "\n".join(block.text for block in result.content if block.type == "text")
        if result.is_error and not text.startswith("Error"):
            text = f"Error: {text}"  # same convention as ToolRegistry.run()
        return text


def load_mcp_tools(connection: MCPConnection) -> List[Tool]:
    return [MCPTool(connection, t) for t in connection.list_tools()]

In [17]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        print(f"summarize:\n{self.history()}")

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [18]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [19]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        pass

In [20]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [21]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-5-5"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        return AssistantMessage(text, tool_calls)

In [22]:
class EmbeddingProvider(ABC):
    model: str = ""  # name of the embedding model, saved together with an index

    @abstractmethod
    def embed(self, texts: List[str]) -> np.ndarray:
        """Return one vector per text, shape (len(texts), dimension)."""
        pass

In [23]:
class OpenAIEmbedding(EmbeddingProvider):

    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.model = model

    def embed(self, texts: List[str]) -> np.ndarray:
        response = self.client.embeddings.create(model=self.model, input=texts)
        data = sorted(response.data, key=lambda d: d.index)  # one embedding per input, in order
        return np.array([d.embedding for d in data])

In [24]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between every row of a and every row of b, shape (len(a), len(b))."""
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

In [25]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, registry: ToolRegistry = None, max_steps: int = 5):
        self.provider = provider
        self.system = system
        self.registry = registry or ToolRegistry()
        self.max_steps = max_steps
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def run(self, message: str, system: SystemMessage = None) -> AssistantMessage:
        system = system or self.system
        tools = self.registry.list()
        turn = [UserMessage(message)]

        response = self.provider.call(system, self.memory.messages + turn, tools)
        turn.append(response)

        step = 0
        while response.tool_calls and step < self.max_steps:
            step += 1
            for call in response.tool_calls:
                result = self.registry.run(call)
                print(f"TOOL[{step}]: {call} -> {result}")
                turn.append(ToolResultMessage(call.id, call.name, result))
            response = self.provider.call(system, self.memory.messages + turn, tools)
            turn.append(response)

        if response.tool_calls:
            response.tool_calls = []
            response.content = response.content or f"(stopped after {self.max_steps} steps)"

        self.memory.add(*turn)
        self.store.save(self.memory)
        return response

    def chat(self, system: SystemMessage = None):
        print(f"Memory #{self.memory.id}")
        while True:
            message = input("YOU: ")
            print(f"YOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                print(f"New memory #{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    print(f"Memory {message} not found")
                else:
                    self.memory = memory
                    print(f"Loaded memory #{memory.id}:\n{memory.history()}")
                continue

            response = self.run(message, system)
            print(f"AI: {response.content}")

In [26]:
class Chunk:

    def __init__(self, text: str, source: str):
        self.text = text
        self.source = source  # which document it comes from, used for citations in chapter 17

    def to_dict(self) -> dict:
        return {"text": self.text, "source": self.source}


def load_documents(folder: str) -> List[tuple]:
    """Return (source, text) for every Markdown file in the folder."""
    return [(path.name, path.read_text(encoding="utf-8")) for path in sorted(Path(folder).glob("*.md"))]

In [27]:
def chunk_fixed(text: str, size: int = 200, overlap: int = 50) -> List[str]:
    """Cut the text every `size` characters; neighbouring chunks share `overlap` characters."""
    step = size - overlap
    return [text[start:start + size] for start in range(0, max(len(text) - overlap, 1), step)]

In [28]:
def chunk_by_heading(text: str, source: str, max_size: int = 500) -> List[Chunk]:
    """One chunk per "## " section, prefixed with its headings; long sections are cut again."""
    chunks, title, heading, lines = [], "", "", []

    def flush():
        body = "\n".join(lines).strip()
        if body:
            prefix = " > ".join(h for h in (title, heading) if h)
            pieces = chunk_fixed(body, max_size, overlap=50) if len(body) > max_size else [body]
            chunks.extend(Chunk(f"{prefix}\n{piece}", source) for piece in pieces)
        lines.clear()

    for line in text.splitlines():
        if line.startswith("# "):
            flush()
            title, heading = line[2:].strip(), ""
        elif line.startswith("## "):
            flush()
            heading = line[3:].strip()
        else:
            lines.append(line)
    flush()
    return chunks

In [29]:
class VectorIndex:

    def __init__(self, embedder: EmbeddingProvider):
        self.embedder = embedder
        self.chunks: List[Chunk] = []
        self.vectors = None  # shape (len(chunks), dimension)

    def add(self, chunks: List[Chunk], batch_size: int = 100):
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
        vectors = np.vstack([self.embedder.embed([c.text for c in batch]) for batch in batches])
        self.vectors = vectors if self.vectors is None else np.vstack([self.vectors, vectors])
        self.chunks += chunks

    def search(self, query: str, top_k: int = 3) -> List[tuple]:
        if not self.chunks:
            return []
        scores = cosine_similarity(self.embedder.embed([query]), self.vectors)[0]
        best = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in best]

    def save(self, folder: str):
        folder = Path(folder)
        folder.mkdir(exist_ok=True)
        np.save(folder / "vectors.npy", self.vectors)
        data = {"model": self.embedder.model, "chunks": [c.to_dict() for c in self.chunks]}
        (folder / "chunks.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, folder: str, embedder: EmbeddingProvider) -> "VectorIndex":
        folder = Path(folder)
        data = json.loads((folder / "chunks.json").read_text(encoding="utf-8"))
        if data["model"] != embedder.model:  # vectors of different models can not be compared
            raise ValueError(f"index was built with '{data['model']}', not '{embedder.model}'")
        index = cls(embedder)
        index.chunks = [Chunk(**c) for c in data["chunks"]]
        index.vectors = np.load(folder / "vectors.npy")
        return index

In [30]:
# the knowledge base was written into knowledge/ in chapter 16
assert Path("knowledge/mcp.md").exists(), "please run chapter 16 first"

In [31]:
%%writefile knowledge/this-book.md
# 本书的 Agent

## 命令
在本书的 Agent 对话中，输入 /quit 退出，输入 /new 开始一段新的记忆，输入 # 加记忆 ID（例如 #a1b2c3d4）加载以前保存的记忆。

## 记忆的保存
每一轮对话结束后，Agent 会自动把记忆保存到 memories 目录下，文件名是记忆 ID 加上 .json 后缀。记忆 ID 是 8 位十六进制字符。

## 工具调用的步数
Agent 每轮对话最多执行 5 步工具调用（max_steps=5）。超过以后，Agent 会停止调用工具，并回复 "(stopped after 5 steps)"。

## MCP 服务器
第 12 章编写的 MCP 服务器代码保存在 mcp_servers 目录中，包括手写的 mcp_server_raw.py 和用官方 SDK 编写的 mcp_server_sdk.py。


Writing knowledge/this-book.md


In [32]:
embedder = OpenAIEmbedding()
index = VectorIndex(embedder)
for source, text in load_documents("knowledge"):
    index.add(chunk_by_heading(text, source))
index.save("index")
print(len(index.chunks), "chunks from", sorted({c.source for c in index.chunks}))

16 chunks from ['mcp.md', 'memory.md', 'this-book.md', 'tool-calling.md']


In [33]:
RAG_PROMPT = """Answer the question using only the context below.
Cite the sources you used, like [mcp.md].
If the context does not contain the answer, say that you don't know.

Context:
{context}

Question: {question}"""


def rag_answer(question: str, index: VectorIndex, provider: LLMProvider, top_k: int = 3) -> str:
    results = index.search(question, top_k)                                       # 1. retrieve
    context = "\n\n".join(f"[{chunk.source}]\n{chunk.text}" for chunk, _ in results)
    prompt = RAG_PROMPT.format(context=context, question=question)                 # 2. augment
    return provider.call(system_message, [UserMessage(prompt)]).content            # 3. generate

In [34]:
provider = OpenAIProvider()
question = "在本书的 Agent 里，怎样切换回以前的一段对话？"

print("Without RAG:\n", provider.call(system_message, [UserMessage(question)]).content)
print("\nWith RAG:\n", rag_answer(question, index, provider))

Without RAG:
 在本书的 Agent 里，**切换回以前的一段对话**通常有两种方式：

1. **使用会话/线程（thread）切换**
   - 如果系统支持多轮会话保存，你可以在界面里打开历史对话列表，直接点选之前的那一段对话。
   - 选中后，Agent 会把那一段对话作为当前上下文继续回答。

2. **通过引用历史消息恢复上下文**
   - 有些 Agent 不一定提供“切换”按钮，但你可以把之前那段对话的关键内容复制出来，贴到当前对话里。
   - 例如你可以说：  
     “请基于下面这段之前的对话继续：……”  
   - 这样 Agent 就会按你提供的历史内容继续推理。

如果你说的“本书的 Agent”是某个具体产品、课程或书里的系统，我也可以按那个系统的界面/代码实现方式，告诉你**具体怎么切换**。你可以把书名或相关截图/说明发我。

With RAG:
 在本书的 Agent 里，可以在输入框中输入 `#` 加记忆 ID（例如 `#a1b2c3d4`）来加载并切换回以前保存的对话。[this-book.md]


In [35]:
for question in ["MCP 有哪些传输方式？", "蒙特利尔一月有多冷？"]:  # the second one is not in the knowledge base
    print("Q:", question)
    print(rag_answer(question, index, provider), "\n")

Q: MCP 有哪些传输方式？
MCP 支持两种主要的传输方式：

1. **stdio**：客户端把服务器作为子进程启动，通过标准输入输出通信，适合本地工具。  
2. **Streamable HTTP**：服务器作为独立的网络服务运行，客户端通过 HTTP POST 发送请求，适合远程部署。  

早期的 **HTTP+SSE** 传输方式已被 **Streamable HTTP** 取代。 [mcp.md] 

Q: 蒙特利尔一月有多冷？
我不知道。给出的上下文没有包含蒙特利尔一月气温的信息。[mcp.md] 



In [36]:
@tool
def search_knowledge(query: Annotated[str, "What to look up, as a short question or keywords"]) -> str:
    """Search the knowledge base about this book: tool calling, MCP, memory and the book's own agent.
    Returns the most relevant passages together with their source file."""
    results = index.search(query, top_k=3)
    return "\n\n".join(f"[{chunk.source}] (score {score:.2f})\n{chunk.text}" for chunk, score in results)


print(search_knowledge("max_steps"))

[this-book.md] (score 0.47)
本书的 Agent > 工具调用的步数
Agent 每轮对话最多执行 5 步工具调用（max_steps=5）。超过以后，Agent 会停止调用工具，并回复 "(stopped after 5 steps)"。

[tool-calling.md] (score 0.43)
工具调用 > 工具循环
一个问题可能需要多次工具调用。Agent 在一个循环中反复调用模型：只要模型还在请求工具，就执行工具并把结果发回，直到模型给出最终回答。为了防止无限循环，通常会设置最大步数。

[tool-calling.md] (score 0.23)
工具调用 > 错误处理
工具执行出错时，不应该让程序崩溃，而应该把错误信息作为工具结果返回给模型。模型看到错误后，往往能够修正参数并重新调用。


In [37]:
rag_system = SystemMessage(
    "You're an AI assistant. For questions about this book or its agent, "
    "use search_knowledge first and cite the sources you used, like [mcp.md]."
)
registry = ToolRegistry([add_days, search_knowledge])
agent = Agent(provider, rag_system, registry=registry)

response = agent.run("本书的 Agent 一轮对话最多能调用几次工具？")
print("AI:", response.content)

TOOL[1]: search_knowledge({"query": "本书 Agent 一轮对话最多能调用几次工具？ tool calls per turn maximum"}) -> [this-book.md] (score 0.77)
本书的 Agent > 工具调用的步数
Agent 每轮对话最多执行 5 步工具调用（max_steps=5）。超过以后，Agent 会停止调用工具，并回复 "(stopped after 5 steps)"。

[tool-calling.md] (score 0.65)
工具调用 > 工具循环
一个问题可能需要多次工具调用。Agent 在一个循环中反复调用模型：只要模型还在请求工具，就执行工具并把结果发回，直到模型给出最终回答。为了防止无限循环，通常会设置最大步数。

[tool-calling.md] (score 0.53)
工具调用 > 什么是工具调用
大模型本身只能生成文本，无法直接访问外部世界。工具调用（Tool Calling，也称 Function Calling）让模型可以在回复中请求调用某个函数：模型输出函数名和参数，由我们的程序真正执行函数，再把结果发回给模型。模型从不直接执行代码。
AI: 本书的 Agent 一轮对话最多能调用 **5 次工具**（`max_steps=5`）。超过后会停止继续调用，并返回 `"(stopped after 5 steps)"`。[this-book.md]


In [38]:
agent.chat()

Memory #ac4aeee2
YOU: /quit
